In [1]:
print("hello world")

hello world


In [2]:
pip install pandas openpyxl psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
Pipeline: CSV → Excel → PostgreSQL
Database : ds360_final_assignment
Table    : ecommerce_customer_behavior
Author   : generated for Wasit
"""

# ── 0. Install dependencies (run once) ───────────────────────────────────────
# pip install pandas openpyxl psycopg2-binary sqlalchemy

import pandas as pd
from sqlalchemy import create_engine, text

# ── 1. CONFIG ─────────────────────────────────────────────────────────────────
CSV_PATH   = r"D:\My Projects\Final Assignment DS360\Dataset & problem Statement\ecommerce_customer_behavior_dataset.csv"
EXCEL_PATH = r"D:\My Projects\Final Assignment DS360\Dataset & problem Statement\ecommerce_customer_behavior.xlsx"

DB_HOST    = "localhost"
DB_PORT    = 5432
DB_NAME    = "ds360_final_assignment"
DB_USER    = "postgres"
DB_PASS    = "wasitheanalyst"
TABLE_NAME = "ecommerce_customer_behavior"

# ── 2. READ CSV ───────────────────────────────────────────────────────────────
print("Reading CSV...")
df = pd.read_csv(CSV_PATH)
print(f"  Loaded {len(df):,} rows × {len(df.columns)} columns")

# ── 3. SAVE TO EXCEL ──────────────────────────────────────────────────────────
print(f"Saving Excel → {EXCEL_PATH}...")
df.to_excel(EXCEL_PATH, index=False, engine="openpyxl")
print("  Excel file saved.")

# ── 4. READ BACK FROM EXCEL ───────────────────────────────────────────────────
print("Reading back from Excel...")
df = pd.read_excel(EXCEL_PATH, engine="openpyxl")
print(f"  Confirmed {len(df):,} rows loaded from Excel.")

# ── 5. CLEAN COLUMN NAMES (PostgreSQL-friendly) ───────────────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\(\)\-\/]+", "_", regex=True)  # spaces/brackets → _
    .str.replace(r"[^\w]", "", regex=True)             # remove any other symbols
    .str.strip("_")
)
print("  Cleaned column names:")
for col in df.columns:
    print(f"    {col}")

# ── 6. CONNECT TO POSTGRESQL ──────────────────────────────────────────────────
print(f"\nConnecting to PostgreSQL → {DB_NAME}...")
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(f"  Connected! {result.fetchone()[0]}")

# ── 7. LOAD INTO POSTGRESQL ───────────────────────────────────────────────────
print(f"\nLoading data into table '{TABLE_NAME}'...")
df.to_sql(
    name=TABLE_NAME,
    con=engine,
    if_exists="replace",   # DROP + re-create table if it already exists
    index=False,
    chunksize=500,          # insert in batches of 500 rows
    method="multi"
)
print(f"  Done! {len(df):,} rows inserted into '{TABLE_NAME}'.")

# ── 8. QUICK VERIFICATION ─────────────────────────────────────────────────────
with engine.connect() as conn:
    count = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).fetchone()[0]
    print(f"\nVerification: SELECT COUNT(*) FROM {TABLE_NAME} → {count:,} rows")

print("\nAll done! Your data is live in PostgreSQL.")

Reading CSV...
  Loaded 10,000 rows × 16 columns
Saving Excel → D:\My Projects\Final Assignment DS360\Dataset & problem Statement\ecommerce_customer_behavior.xlsx...
  Excel file saved.
Reading back from Excel...
  Confirmed 10,000 rows loaded from Excel.
  Cleaned column names:
    customer_id
    age
    gender
    location
    product_category
    purchase_amount
    time_spent_on_website_min
    device_type
    payment_method
    discount_availed
    number_of_items_purchased
    return_customer
    review_score_1_5
    delivery_time_days
    subscription_status
    customer_satisfaction

Connecting to PostgreSQL → ds360_final_assignment...
  Connected! PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit

Loading data into table 'ecommerce_customer_behavior'...
  Done! 10,000 rows inserted into 'ecommerce_customer_behavior'.

Verification: SELECT COUNT(*) FROM ecommerce_customer_behavior → 10,000 rows

All done! Your data is live in PostgreSQL.
